# Experimentación Computacional en *Machine Learning* (Parte 2)


## Validación Cruzada Estratificada ($k$-Fold) (`Breast Cancer`)

### [Wikipedia - Validación Cruzada](https://es.wikipedia.org/wiki/Validación_cruzada)





* ### Cuando evaluamos un modelo mediante una única partición (*hold-out*), la métrica obtenida depende fuertemente de qué observaciones cayeron en el conjunto de evaluación. 
* ### Para obtener una estimación más estable y reducir la varianza debida al particionamiento aleatorio, empleamos **Validación Cruzada Estratificada de $k$ particiones o *folds* (*Stratified $k$-Fold Cross-Validation*)**.
* ### En este esquema, el conjunto de datos se divide en $k$ subconjuntos disjuntos de tamaño aproximado $D_1, D_2, \dots, D_k$, conservando en cada partición la proporción original de las clases ($P(Y=c \mid D_i) \approx P(Y=c \mid D)$). 
* ### En cada iteración $i \in \{1, \dots, k\}$, el modelo se entrena con $k-1$ particiones ($D \setminus D_i$) y se evalúa sobre la partición restante ($D_i$), obteniendo $k$ mediciones pareadas de desempeño ($\mu \pm \sigma$).

### En esta sección implementaremos:
### 1. **Configuración de `StratifiedKFold`:** definición de $k=10$ particiones estratificadas fijando una semilla (`random_state`) para garantizar reproducibilidad.
### 2. **Verificación de balance por partición:** comprobación explícita de que cada partición conserva la proporción de clases (`Maligno` vs. `Benigno`).
### 3. **Evaluación multi-métrica por partición:** cálculo de `Accuracy`, `Precision`, `Recall`, `F1-score macro` en cada iteración para reportar el desempeño promedio y su desviación estándar ($\mu \pm \sigma$), así como guardar todas las evaluaciones.

![Esquema validación cruzada](https://upload.wikimedia.org/wikipedia/commons/1/18/Esquema_castellà.jpg?utm_source=es.wikipedia.org&utm_campaign=imageinfo&utm_content=original)


# 1. Cargar el dataset `Breast Cancer`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo
import pandas as pd
from ucimlrepo import fetch_ucirepo

brest_cancer = fetch_ucirepo(id=17)
X = brest_cancer.data.features
y = brest_cancer.data.targets
serie_y = y["Diagnosis"]


# Calcular distribución de la variable objetivo ('Diagnosis')

conteo = serie_y.value_counts()
proporcion = serie_y.value_counts(normalize=True)* 100

distribucion = pd.DataFrame(
    {
        "Calidad (Clase)": conteo.index,
        "Frecuencia (n)": conteo.values,
        "Porcentaje (%)": proporcion.values.round(2),
    }
)

print(f"Total de muestras: {len(y)}")
print("\nDistribución de clases (Diagnosis):")
print(distribucion.to_string(index=False))

## 2. Validación Cruzada Estratificada ($k$-Fold Cross-Validation, $k=10$)


### [Documentación oficial StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)


### Para evitar que la evaluación dependa de una sola partición aleatoria y reducir la varianza experimental, dividimos el dataset en **$k=10$ particiones estratificados** conservando la proporción de clases en cada iteración:
### 1. **Particionamiento estratificado:** Configuramos `StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)` para evaluar 10 bloques pareados (90% entrenamiento y 10% prueba en cada fold).
### 2. **Preprocesamiento aislado por fold:** En cada iteración ajustamos el escalador (`StandardScaler.fit_transform`) estrictamente sobre `X_train` y aplicamos esos parámetros (`transform`) sobre `X_test`.
### 3. **Evaluación pareada ($\mu \pm \sigma$):** Entrenamos el clasificador (`SVC`) y calculamos `Accuracy`, `Precision`, `Recall`  y `F1-Macro` en los 10 folds para obtener una estimación robusta del desempeño.

> ### **Regla experimental:** Nunca se debe aplicar el escalador sobre el dataset completo antes de dividir los folds; hacerlo introduce **fuga de datos (*Data Leakage*)** e infla artificialmente las métricas.

# 3. Entrenamiento de un clasificador SVM (`SVC`)

* ### Documentación oficial **SVC**: [sklearn.svm.SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)

* ### **Support Vector Machine (`SVC`)** es un algoritmo de aprendizaje supervisado que busca el hiperplano óptimo capaz de separar las clases maximizando el margen de distancia hacia las observaciones frontera más cercanas (vectores de soporte), utilizando funciones *kernel* (como `rbf`) para proyectar fronteras no lineales.

* ### Aspectos clave para su evaluación experimental:
    ### 1. **Sensibilidad a la escala:** al optimizar márgenes geométricos en el espacio de características, variables con rangos numéricos amplios distorsionan la orientación del hiperplano.
    ### 2. **Regularización y *Kernel* (`C`, `kernel='rbf'`):** controlan la flexibilidad de la frontera de decisión y la tolerancia a errores de clasificación en entrenamiento.
    ### 3. **Condición pareada:** sirve como primer tratamiento de referencia ($\text{Grupo 1}$); sus $k$ mediciones por fold (`SEED = 42`) forman la línea base contra la cual se contrastarán los demás modelos.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

SEED = 42  # Semilla fija para garantizar reproducibilidad


# -------------------------------------------------------------------------
# 1. Configuración de Stratified 10-Fold Cross-Validation
# -------------------------------------------------------------------------
# n_splits=10: divide los datos en 10 particiones (90% train, 10% test por iteración)
# shuffle=True: mezcla los datos antes de partirlos
# random_state=42: garantiza que las particiones sean exactamente reproducibles
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# Lista para almacenar los resultados obtenidos en cada uno de los 10 folds o particiones
resultados_svm = []

print("Ejecutando Validación Cruzada Estratificada (10-Fold) paso a paso:\n")

# -------------------------------------------------------------------------
# 2. Ciclo explícito sobre cada partición (fold)
# -------------------------------------------------------------------------
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

    # A. Separación física de entrenamiento (9 folds) y test (1 fold)
    # .iloc selecciona las filas por posición entera (0, 1, 2...) usando los índices generados por StratifiedKFold
    X_train = X.iloc[train_idx]  # Extrae las filas asignadas al conjunto de entrenamiento (90%)
    X_test = X.iloc[test_idx]    # Extrae las filas asignadas al conjunto de prueba de este fold (10%)

    X_train = X.iloc[train_idx].to_numpy()
    X_test = X.iloc[test_idx].to_numpy()


    # Aplanamos y_train y y_test de (n, 1) a un arreglo 1D (n,)
    y_train = y.iloc[train_idx].to_numpy().ravel()
    y_test = y.iloc[test_idx].to_numpy().ravel()

    # B. Preprocesamiento SIN fuga de datos (Data Leakage):
    #    - Instanciamos un escalador nuevo en cada fold.
    #    - Calculamos media y desviación estándar ÚNICAMENTE con X_train (fit_transform).
    #    - Aplicamos esos mismos parámetros a X_test (transform), sin volver a ajustar.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # D. Instanciación y entrenamiento del clasificador SVM
    modelo = SVC(kernel="rbf", random_state=42)
    modelo.fit(X_train_scaled, y_train)

    # E. Generación de predicciones tanto en entrenamiento como en validación
    y_train_pred = modelo.predict(X_train_scaled)
    y_test_pred = modelo.predict(X_test_scaled)

    # F. Cálculo de métricas de desempeño en este pliegue

    train_f1 = f1_score(y_train, y_train_pred, average="macro")
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average="macro")
    test_precision = precision_score(y_test, y_test_pred, average="macro")
    test_recall = recall_score(y_test, y_test_pred, average="macro")

    # G. Guardamos las métricas y estadísticas de balance de este fold
    resultados_svm.append(
        {
            "Fold": fold,
            "Train_F1_Macro": train_f1,
            "Test_Accuracy": test_acc,
            "Test_F1_Macro": test_f1,
            "Test_Precision": test_precision,
            "Test_Recall": test_recall
        }
    )

    print(f"Fold: {fold} Test F1-Macro: {test_f1:.4f}")

# -------------------------------------------------------------------------
# 3. Consolidación de resultados en un DataFrame y resumen estadístico
# -------------------------------------------------------------------------
df_folds = pd.DataFrame(resultados_svm)

# Muestra el dataframe
# display(df_folds)

# Cálculo de media y desviación estándar muestral (ddof=1) a través de los 10 folds
media_acc = df_folds["Test_Accuracy"].mean()
std_acc = df_folds["Test_Accuracy"].std(ddof=1)

media_f1 = df_folds["Test_F1_Macro"].mean()
std_f1 = df_folds["Test_F1_Macro"].std(ddof=1)

print(f"\nResumen Global (10 Folds):")
print(f"Accuracy Test : {media_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Macro Test : {media_f1:.4f} ± {std_f1:.4f}")

# 3. Entrenamiento de un clasificador KNN

* ### **$k$-Nearest Neighbors ($k$-NN)** es un algoritmo de aprendizaje supervisado basado en instancias (aprendizaje perezoso o *lazy learning*) que clasifica cada nueva observación asignándole la clase mayoritaria entre sus $k$ vecinos más cercanos en el espacio de características, medidos habitualmente mediante distancia euclidiana ($d(x, x') = \sqrt{\sum_{j=1}^{p}(x_j - x'_j)^2}$).

* ### Aspectos clave para su evaluación experimental:
    ### 1. **Sensibilidad a la escala:** variables con mayor magnitud pueden dominar  el cálculo de la distancia.
    ### 2. **Hiperparámetro $k$ (`n_neighbors`):** Controla el equilibrio entre sesgo y varianza; valores muy pequeños hacen al modelo sensible al ruido, mientras que valores muy grandes suavizan excesivamente las fronteras de decisión.
    ### 3. **Condición pareada:** Para compararlo válidamente frente a `SVM`, debe entrenarse y evaluarse exactamente sobre las mismas particiones (`SEED = 42`) y con el mismo preprocesamiento dentro de cada fold.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

SEED = 42  # Semilla fija para garantizar reproducibilidad


# -------------------------------------------------------------------------
# 1. Configuración de Stratified 10-Fold Cross-Validation
# -------------------------------------------------------------------------
# n_splits=10: divide los datos en 10 particiones (90% train, 10% test por iteración)
# shuffle=True: mezcla los datos antes de partirlos
# random_state=42: garantiza que las particiones sean exactamente reproducibles
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# Lista para almacenar los resultados obtenidos en cada uno de los 10 folds o particiones
resultados_knn = []

print("Ejecutando Validación Cruzada Estratificada (10-Fold) paso a paso:\n")

# -------------------------------------------------------------------------
# 2. Ciclo explícito sobre cada partición (fold)
# -------------------------------------------------------------------------
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

    # A. Separación física de entrenamiento (9 folds) y test (1 fold)
    # .iloc selecciona las filas por posición entera (0, 1, 2...) usando los índices generados por StratifiedKFold
    X_train = X.iloc[train_idx].to_numpy()  # Extrae las filas asignadas al conjunto de entrenamiento (90%)
    X_test = X.iloc[test_idx].to_numpy()  # Extrae las filas asignadas al conjunto de prueba de este fold (10%)


    # Aplanamos y_train y y_test de (n, 1) a un arreglo 1D (n,)
    y_train = y.iloc[train_idx].to_numpy().ravel()
    y_test = y.iloc[test_idx].to_numpy().ravel()

    # B. Preprocesamiento SIN fuga de datos (Data Leakage):
    #    - Instanciamos un escalador nuevo en cada fold.
    #    - Calculamos media y desviación estándar ÚNICAMENTE con X_train (fit_transform).
    #    - Aplicamos esos mismos parámetros a X_test (transform), sin volver a ajustar.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # D. Instanciación y entrenamiento del clasificador K-Nearest Neighbors (k=5 vecinos por defecto)
    modelo_knn = KNeighborsClassifier(n_neighbors=5)
    modelo_knn.fit(X_train_scaled, y_train)
    y_test_pred = modelo_knn.predict(X_test_scaled)

    # E. Generación de predicciones tanto en entrenamiento como en validación
    y_train_pred = modelo.predict(X_train_scaled)
    y_test_pred = modelo.predict(X_test_scaled)

    # F. Cálculo de métricas de desempeño en este pliegue

    train_f1 = f1_score(y_train, y_train_pred, average="macro")
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average="macro")
    test_precision = precision_score(y_test, y_test_pred, average="macro")
    test_recall = recall_score(y_test, y_test_pred, average="macro")

    # G. Guardamos las métricas y estadísticas de balance de este fold
    resultados_knn.append(
        {
            "Fold": fold,
            "Train_F1_Macro": train_f1,
            "Test_Accuracy": test_acc,
            "Test_F1_Macro": test_f1,
            "Test_Precision": test_precision,
            "Test_Recall": test_recall,
        }
    )

    print(f"Fold: {fold} Test F1-Macro: {test_f1:.4f}")

# -------------------------------------------------------------------------
# 3. Consolidación de resultados en un DataFrame y resumen estadístico
# -------------------------------------------------------------------------
df_folds_knn = pd.DataFrame(resultados_knn)

# Muestra el dataframe
# display(df_folds_knn)

# Cálculo de media y desviación estándar muestral (ddof=1) a través de los 10 folds
media_acc = df_folds_knn["Test_Accuracy"].mean()
std_acc = df_folds_knn["Test_Accuracy"].std(ddof=1)

media_f1 = df_folds_knn["Test_F1_Macro"].mean()
std_f1 = df_folds_knn["Test_F1_Macro"].std(ddof=1)

print(f"\nResumen Global (10 Folds):")
print(f"Accuracy Test : {media_acc:.4f} ± {std_acc:.4f}")
print(f"F1-Macro Test : {media_f1:.4f} ± {std_f1:.4f}")

## 4. Comparación Pareada de Dos Clasificadores (`SVM` vs. `KNN` — Análisis Intra-Dominio)

* ### Para determinar si la diferencia de desempeño entre dos algoritmos es estadísticamente significativa y no producto del azar, evaluamos ambos modelos exactamente sobre los **mismos 10 folds estratificados** (`SEED = 42`), generando una **matriz pareada de $2 \text{ grupos/filas (algoritmos)} \times 10 \text{ columnas (folds)}$**:
    ### 1. **Evaluación bajo igualdad de condiciones:** Entrenamos `KNeighborsClassifier` sobre las mismas particiones y con el mismo escalamiento (`StandardScaler`); de forma similar, se aplicó al clasificador  `SVM`.
    ### 2. **Prueba de normalidad sobre las diferencias ($d_i = \text{F1}_{\text{SVM}, i} - \text{F1}_{\text{KNN}, i}$):** Aplicamos la prueba de **Shapiro-Wilk** ($n=10$) para verificar si las diferencias pareadas siguen una distribución normal ($p > 0.05$).
    ### 3. **Contraste de hipótesis pareado:** Aplicamos la **prueba $t$ de Student pareada** (si hay normalidad) o la **prueba de rangos con signo de Wilcoxon** (si no hay normalidad) y reportamos el valor $p$ exacto junto con el tamaño del efecto.

> ### **Regla experimental:** Dos algoritmos solo pueden compararse mediante pruebas pareadas si fueron evaluados exactamente sobre las **mismas particiones de datos** en cada fold.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# -------------------------------------------------------------------------
# 1. Consolidación de resultados y construcción de los grupos
# -------------------------------------------------------------------------
# Convertimos las listas de diccionarios (resultados por fold) en DataFrames de pandas
df_svm = pd.DataFrame(resultados_svm)
df_knn = pd.DataFrame(resultados_knn)

# Definimos la métrica objetivo a evaluar
metrica = "Test_F1_Macro"

# Extraemos la columna de la métrica como arreglos 1D de NumPy (10 valores pareados por algoritmo)
scores_svm = df_svm[metrica].to_numpy()
scores_knn = df_knn[metrica].to_numpy()

# Imprimimos ambos vectores para verificar que tienen la misma longitud (n=10) y están alineados por fold
print(scores_svm)
print(scores_knn)

In [ ]:
diferencias = scores_svm - scores_knn  # Vector de 10 diferencias pareadas (d_i)


# -------------------------------------------------------------------------
# 2. Verificación de Supuestos: Normalidad (Shapiro-Wilk) y Varianzas (Levene)
# -------------------------------------------------------------------------
# A. Normalidad sobre el vector de diferencias pareadas (supuesto crítico para t-pareada)

w_dif, p_norm_dif = stats.shapiro(diferencias)
print(f"Estadistico W:{w_dif}")
print(f"P-valor Shapiro:{p_norm_dif}")

stat_levene, p_levene = stats.levene(scores_svm, scores_knn)
print(f"Prueba de Levene (Varianza): F = {stat_levene:.4f} | p = {p_levene:.4f}")

In [ ]:

print("\n=== VERIFICACIÓN DE SUPUESTOS (alpha = 0.05) ===")
print(
    f"Shapiro-Wilk Diferencias  : W = {w_dif:.4f} | p = {p_norm_dif:.4f} ({'Normal' if p_norm_dif > 0.05 else 'No normal'})"
)
print(
    f"Prueba de Levene (Varianza): F = {stat_levene:.4f} | p = {p_levene:.4f} ({'Homocedasticidad' if p_levene > 0.05 else 'Heterocedasticidad'})"
)

In [ ]:
# -------------------------------------------------------------------------
# 3. Diagnóstico Visual: Q-Q Plots (SVM, KNN y Diferencias Pareadas)
# -------------------------------------------------------------------------

plt.figure(figsize=(8, 6))
titulo = f"Q-Q Plot: Diferencias (SVM - KNN)\n(Shapiro p = {p_norm_dif:.4f})"
stats.probplot(diferencias, dist="norm", plot=plt)
plt.title(titulo, fontsize=10, fontweight="bold")
plt.xlabel("Cuantiles teóricos")
plt.ylabel("Cuantiles muestrales")
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


# Formulación de Hipótesis (Prueba Unilateral de Rangos con Signo de Wilcoxon)


### Justificación Teórica *a priori* para la Hipótesis Unilateral (`SVM > KNN`)

### Para justificar un contraste unilateral derecho (`alternative="greater"`, donde $d_i = \text{F1}_{\text{SVM}, i} - \text{F1}_{\text{KNN}, i}$) **antes de observar los resultados**, nos apoyamos en las propiedades geométricas del problema (*Breast Cancer*, $p = 30$ características continuas):

### 1. **El problema de `KNN` con muchas variables:** `KNN` mide distancias dándole exactamente el mismo peso a las 30 características. Cuando hay tantas variables —y varias están repetidas o aportan poco—, todos los casos parecen estar a distancias muy parecidas, haciendo más borrosa la frontera entre clases.
### 2. **La ventaja de `SVM` en la frontera:** `SVM` busca trazar la frontera con el mayor margen de seguridad posible enfocándose únicamente en los casos más difíciles del límite (vectores de soporte), por lo que ignora el ruido del resto de los datos y generaliza mejor con 30 variables.

### Planteamiento Formal de Hipótesis ($\alpha = 0.05$)

* #### **Hipótesis Nula ($H_0$):** $\tilde{d} \le 0$ — El desempeño mediano de `SVM` es menor o igual al de `KNN` a través de los 10 folds.
* #### **Hipótesis Alternativa ($H_1$):** $\tilde{d} > 0$ — El desempeño mediano de `SVM` es estrictamente superior al de `KNN` debido a su mejor manejo de fronteras en espacios de 30 dimensiones.


> #### **Criterio de decisión:** Si $p\text{-valor} < 0.05$, se rechaza $H_0$ y se concluye que `SVM` supera estadísticamente a `KNN`; si $p\text{-valor} \ge 0.05$, no existe evidencia estadística suficiente para afirmar que `SVM` sea superior.

In [ ]:
# -------------------------------------------------------------------------
# 4. Contraste de Hipótesis Pareado y Tamaño del Efecto
# -------------------------------------------------------------------------
alpha = 0.05
print("\n=== CONTRASTE DE HIPÓTESIS PAREADO ===")

stat_test, p_val = stats.wilcoxon(scores_svm, scores_knn, alternative="greater")


print(f"Estadístico         : {stat_test:.4f}")
print(f"p-Valor             : {p_val:.6f}")



## Interpretación de Resultados del Contraste Pareado

* ### **Lectura del estadístico ($W^+ = 0.0000$):** En el contraste unilateral derecho (`alternative="greater"`), el estadístico de Wilcoxon suma únicamente los rangos de los folds donde `SVM` supera a `KNN` ($d_i > 0$). Un valor de $0.0000$ indica que **`SVM` no superó a `KNN` en ninguno de los 10 folds**.
* ### **Decisión estadística ($p = 1.000000 \ge 0.05$):** **No se rechaza la hipótesis nula ($H_0: \tilde{d} \le 0$)**. 
* ### No existe evidencia estadística para afirmar que `SVM` tenga un desempeño superior a `KNN`.
* ### **Nota metodológica:** Este caso ilustra la importancia de definir la dirección de una prueba unilateral *a priori* (antes de ver los datos) o emplear un contraste bilateral ($H_1: \tilde{d} \neq 0$) cuando no existe una hipótesis teórica previa sobre qué clasificador dominará en el dominio.

# Guía de Reporte de Resultados del Contraste Pareado

### Para que el análisis estadístico sea riguroso y reproducible, al finalizar el contraste de hipótesis debes reportar los siguientes **elementos**:

### 1. **Estadísticos descriptivos por modelo y de las diferencias ($d_i = \text{Score}_{A,i} - \text{Score}_{B,i}$):**
   * ### **Media y desviación estándar** de cada clasificador ($\bar{X}_A \pm s_A$ vs. $\bar{X}_B \pm s_B$), junto con la **diferencia de medias** ($\bar{d} = \bar{X}_A - \bar{X}_B$).
   * ### Si se utilizó la prueba no paramétrica de **Wilcoxon**, reportar también la **mediana de cada modelo**, la **mediana de las diferencias** ($\tilde{d}$).

### 2. **Prueba aplicada y tipo de contraste (`alternative`):**
   * ### Indicar explícitamente si se ejecutó **$t$ de Student pareada** (`ttest_rel`) o **Wilcoxon de rangos con signo** (`wilcoxon`), justificando la elección con el resultado de Shapiro-Wilk.
   * ### Especificar si el contraste fue **bilateral (`two-sided`)** o **unilateral (`greater` / `less`)** y recordar el orden de resta empleado ($A - B$).



### 3. **Valor del estadístico de prueba y $p$-valor exacto:**
   * ### Reportar el estadístico obtenido ($t$ o $W$) y el $p$-valor exacto comparado contra $\alpha = 0.05$ (nunca escribir únicamente *"significativo"* o *"no significativo"* sin mostrar ambos números).

### 4. **Decisión estadística formal (según el `alternative` usado):**
   * ### **Si $p < 0.05$ (Se rechaza $H_0$):**
     * ### *En contraste bilateral (`two-sided`):* Concluir que **existe una diferencia estadísticamente significativa** entre ambos modelos e identificar al superior mediante el signo de la diferencia de medias ($\bar{d}$) o medianas ($\tilde{d}$).
      > ### **Nota metodológica:** Si el contraste bilateral (`two-sided`) arroja $p < 0.05$, **no se ejecuta una segunda prueba unilateral**. El rechazo de $H_0$ ya confirma que la diferencia es estadísticamente significativa; el modelo superior se identifica directamente comparando la **media** (para $t$ pareada) o la **mediana** (para Wilcoxon) de ambos clasificadores.
            
     * ### *En contraste unilateral (`greater` con $A - B$):* Concluir directamente que **el Modelo $A$ es estadísticamente superior al Modelo $B$**.
   * ### **Si $p \ge 0.05$ (No se rechaza $H_0$):**
     * ### *En contraste bilateral (`two-sided`):* Concluir que **no existe evidencia estadística suficiente** para afirmar que haya diferencia entre ambos clasificadores.
     * ### *En contraste unilateral (`greater` con $A - B$):* Concluir que **no existe evidencia estadística suficiente para afirmar que el Modelo $A$ supere al Modelo $B$** (sin inferir automáticamente la desigualdad contraria).

### 5. **Interpretación experimental (relevancia práctica):**
- ### Más allá de si $p < 0.05$, analiza si la ganancia real en puntos porcentuales ($\bar{d} \times 100\%$) y la estabilidad entre folds (menor desviación estándar) justifican elegir un modelo sobre otro en un entorno real (por ejemplo, si una mejora mínima de $0.3\%$ compensa usar un modelo más lento o complejo).

# Ejercicio: Comparación  (`RandomForest` vs. `KNN`)

### Objetivo
* ### Aplicar el protocolo completo de validación cruzada estratificada ($k = 10$, `SEED = 42`) y contraste de hipótesis pareado para evaluar si existe una diferencia estadísticamente significativa entre un ensamble basado en árboles (`RandomForestClassifier`) y un clasificador basado en distancias (`KNeighborsClassifier`) sobre el conjunto de datos *Breast Cancer*.



### Instrucciones

### 1. **Entrenamiento y evaluación (sin fuga de datos):**
   * ### Utilizando **exactamente las mismas 10 particiones estratificadas** (`StratifiedKFold(n_splits=10, shuffle=True, random_state=42)`), entrena **`RandomForestClassifier(n_estimators=100, random_state=42)`** y compáralo contra **`KNeighborsClassifier(n_neighbors=5)`**.
   * ### **Nota sobre el escalamiento:** Aplica `StandardScaler` ajustado únicamente con `X_train` en cada pliegue para mantener igualdad de condiciones en el flujo.
   * ### Guarda los resultados por fold utilizando como métrica objetivo **`Test_F1_Accuracy`**.

### 2. **Extracción de vectores pareados y diagnóstico de supuestos:**
   * ### Extrae la métrica objetivo **`Test_F1_Accuracy`** como arreglos 1D de NumPy alineados por fold (`scores_rf` y `scores_knn`), y reporta la media y desviación estándar de cada grupo.
   * ### Calcula el vector de diferencias pareadas $d_i = \text{F1}_{\text{RF}, i} - \text{F1}_{\text{KNN}, i}$.
   * ### Ejecuta la prueba de normalidad, la prueba de homogeneidad de varianzas de **Levene** entre ambos grupos y grafica el **Q-Q plot** de las diferencias.

### 3. **Planteamiento de hipótesis y contraste estadístico:**
- ### Dado que **no se tiene preferencia ni hipótesis direccional previa** sobre cuál de los dos algoritmos (`RandomForest` o `KNN`) tendrá mejor desempeño en este dominio, formula $H_0$ y $H_1$ en una celda de Markdown **antes** de ejecutar la prueba, justificando si corresponde un **contraste bilateral (no direccional) o unilateral (direccional)** y seleccionando la prueba adecuada ($t$ de Student pareada o Wilcoxon de rangos con signo) según el resultado de normalidad ($\alpha = 0.05$).


---

### Preguntas de Análisis (Responder en Markdown)

### 1. **Elección del contraste:** Al no existir preferencia ni hipótesis direccional previa sobre cuál algoritmo logrará mayor `Accuracy`, plantea la dirección adecuada.
### 2. **Normalidad y empates en `Accuracy`:** Al evaluar las 10 diferencias pareadas $d_i$ de `Accuracy` (una por cada fold), ¿cumplen el supuesto de normalidad ($p > 0.05$ en Shapiro-Wilk)? 
### 3. **Rigor estadístico:** ¿Por qué no es válido ejecutar primero `alternative="greater"` y luego `alternative="less"` sobre los mismos 10 folds si la primera prueba no resulta significativa?
### 4. **Conclusión experimental:** Con base en el $p$-valor obtenido, la media/mediana de `Accuracy` y su dispersión entre folds, ¿existe evidencia estadística suficiente para afirmar que hay una diferencia real de desempeño entre `RandomForest` y `KNN` en este conjunto de datos?

# **Random Forest (`RandomForestClassifier`)** 

* ### Es un algoritmo de aprendizaje supervisado por ensamble basado en *Bagging* (*Bootstrap Aggregating*) que construye múltiples árboles de decisión independientes sobre submuestras aleatorias con reemplazo de los datos, evaluando en cada división un subconjunto aleatorio de características, y determina la clase final mediante votación mayoritaria ($\hat{y} = \text{moda}\{h_1(x), h_2(x), \dots, h_B(x)\}$).

- ### Aspectos clave para su evaluación experimental:
    ### 1. **Invarianza a la escala:** Al realizar particiones ortogonales basadas en umbrales individuales por variable, no requiere estandarización para su funcionamiento interno 
    ### 2. **Hiperparámetro $B$ (`n_estimators`):** Controla el número de árboles del ensamble; incrementar los estimadores reduce la varianza del modelo y estabiliza las predicciones sin provocar sobreajuste, a costa de un mayor tiempo de cómputo.
    ### 3. **Condición pareada:** Para compararlo válidamente frente a `KNN`, debe entrenarse y evaluarse exactamente sobre las mismas particiones (`SEED = 42`), fijando `random_state=SEED` para garantizar la reproducibilidad del muestreo *bootstrap*.

In [ ]:
# TODO:

from sklearn.ensemble import RandomForestClassifier

# Definición del clasificador Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)